<a href="https://colab.research.google.com/github/ShreyaMathur19/Clustered-Diagonal-Segment-Format-CDSF-/blob/main/HIPC_Submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from scipy.io import mmread
import numpy as np
from scipy.sparse import coo_matrix
from time import perf_counter
import numba as nb

In [ ]:
# Load the matrix from an .mtx file
A = mmread("wang3.mtx")
A = coo_matrix(A)

In [ ]:
print(type(A))   # usually returns <class 'scipy.sparse.coo.coo_matrix'>
print(A.shape)   # dimensions of the matrix
print(A.nnz)     # number of nonzero entries

<class 'scipy.sparse._coo.coo_matrix'>
(26064, 26064)
177168


In [ ]:
# Convert COO to DIA
A_dia = A.todia()

In [ ]:
print(type(A_dia))   # <class 'scipy.sparse.dia.dia_matrix'>
print(A_dia.shape)   # dimensions
print(A_dia.data.shape)    # stored diagonals
print(len(A_dia.offsets)) # diagonal offsets

<class 'scipy.sparse._dia.dia_matrix'>
(26064, 26064)
(21, 26064)
21


In [ ]:
# Memory usage
data_bytes = A_dia.data.nbytes        # storage for diagonal values
offset_bytes = A_dia.offsets.nbytes   # storage for offsets
total_bytes = data_bytes + offset_bytes

print("Shape:", A_dia.shape)
print("Data shape:", A_dia.data.shape)   # (num_diagonals, n)
print("Offsets length:", len(A_dia.offsets))
print("Data memory (bytes):", data_bytes)
print("Offsets memory (bytes):", offset_bytes)
print("Total memory (bytes):", total_bytes)
print(f"Total memory: {total_bytes/(1024*1024):.2f} MB")

Shape: (26064, 26064)
Data shape: (21, 26064)
Offsets length: 21
Data memory (bytes): 4378752
Offsets memory (bytes): 84
Total memory (bytes): 4378836
Total memory: 4.18 MB


In [ ]:
import numpy as np
from scipy.sparse import coo_matrix

def encode_cdsf(A):
    """
    Encode a sparse matrix A into CDSF without a bandwidth parameter.
    Scans all diagonals; clusters are maximal runs of consecutive nonzeros
    along a diagonal (i.e., (i,k),(i+1,k+1),...).

    Returns:
        row_indices : int32  [num_clusters]  start row of each cluster
        col_indices : int32  [num_clusters]  start col of each cluster
        lengths     : int32  [num_clusters]  length (number of values) of each cluster
        values      : float64 [sum(lengths)] concatenated cluster values in order
    """
    # Ensure COO; coalesce duplicates; force float64 values
    A = A.tocoo(copy=False)
    if hasattr(A, "sum_duplicates"):
        A.sum_duplicates()
    row = A.row.astype(np.int32, copy=False)
    col = A.col.astype(np.int32, copy=False)
    data = A.data.astype(np.float64, copy=False)

    nnz = data.size
    if nnz == 0:
        # Empty matrix
        return (np.empty(0, dtype=np.int32),
                np.empty(0, dtype=np.int32),
                np.empty(0, dtype=np.int32),
                np.empty(0, dtype=np.float64))

    # Diagonal offset: col - row
    off = (col - row).astype(np.int32, copy=False)

    # Sort by (offset, row) so each diagonal is contiguous and increasing
    # np.lexsort uses last key as primary; we want offset primary, row secondary.
    order = np.lexsort((row, off))
    off_s  = off[order]
    row_s  = row[order]
    col_s  = col[order]
    data_s = data[order]

    # Detect starts of clusters:
    # A new cluster starts at the first element, or whenever the diagonal changes,
    # or when rows are not consecutive (break in the diagonal run).
    # is_new_cluster[k] = True iff k==0 or off_s[k]!=off_s[k-1] or row_s[k]!=row_s[k-1]+1
    is_new = np.empty(nnz, dtype=bool)
    is_new[0] = True
    same_diag = off_s[1:] == off_s[:-1]
    consec_row = row_s[1:] == (row_s[:-1] + 1)
    is_new[1:] = ~(same_diag & consec_row)

    # Cluster start indices and lengths (run-length encoding)
    starts = np.nonzero(is_new)[0]
    ends   = np.empty_like(starts)
    ends[:-1] = starts[1:]
    ends[-1]  = nnz
    lengths = (ends - starts).astype(np.int32, copy=False)

    # Cluster starting coordinates
    row_indices = row_s[starts].astype(np.int32, copy=False)
    col_indices = col_s[starts].astype(np.int32, copy=False)

    # Values are already in cluster-concatenated order
    values = data_s  # float64

    return row_indices, col_indices, lengths, values


In [ ]:
row_indices, col_indices, lengths, values = encode_cdsf(A)
print(row_indices.size)
print(col_indices.size)
print(lengths.size)
print(values.size)

1815
1815
1815
177168


In [ ]:
def cdsf_memory_bytes(row_indices, col_indices, lengths, values):
    return (row_indices.nbytes + col_indices.nbytes +
            lengths.nbytes + values.nbytes)

def cdsf_memory_report(row_indices, col_indices, lengths, values):
    b = cdsf_memory_bytes(row_indices, col_indices, lengths, values)
    kb = b / 1024
    mb = b / (1024*1024)
    print(f"Arrays:")
    print(f"  row_indices: {row_indices.nbytes/1024:.2f} KB")
    print(f"  col_indices: {col_indices.nbytes/1024:.2f} KB")
    print(f"  lengths    : {lengths.nbytes/1024:.2f} KB")
    print(f"  values     : {values.nbytes/1024:.2f} KB")
    print(f"Total: {kb:.2f} KB  ({mb:.2f} MB)")



In [ ]:
cdsf_memory_report(row_indices, col_indices, lengths, values)

Arrays:
  row_indices: 7.09 KB
  col_indices: 7.09 KB
  lengths    : 7.09 KB
  values     : 1384.12 KB
Total: 1405.39 KB  (1.37 MB)


In [ ]:
matrix_list = ['wang3', 'wang4', 's3dkt3m2', 's3dkq4m2', 'kim1', 'kim2', 'nemeth21', 'nemeth22',
               'crystk02', 'crystk03', 'af_1_k101', 'af_2_k101']
matrix_list


['wang3',
 'wang4',
 's3dkt3m2',
 's3dkq4m2',
 'kim1',
 'kim2',
 'nemeth21',
 'nemeth22',
 'crystk02',
 'crystk03',
 'af_1_k101',
 'af_2_k101']

In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    A = coo_matrix(A)

    # Convert COO to DIA
    A_dia = A.todia()

    # Memory usage
    data_bytes = A_dia.data.nbytes        # storage for diagonal values
    offset_bytes = A_dia.offsets.nbytes   # storage for offsets
    total_bytes = data_bytes + offset_bytes

    print("Shape:", A_dia.shape)
    print("Data shape:", A_dia.data.shape)   # (num_diagonals, n)
    print("Offsets length:", len(A_dia.offsets))
    print(A_dia.offsets)
    print("min offsets",min(A_dia.offsets))
    print("max offsets", max(A_dia.offsets))
    print("Data memory (bytes):", data_bytes)
    print("Offsets memory (bytes):", offset_bytes)
    print("Total memory (bytes):", total_bytes)
    print(f"Total memory: {total_bytes/(1024*1024):.2f} MB")

displaying info for wang3
------------------------------
Shape: (26064, 26064)
Data shape: (21, 26064)
Offsets length: 21
[-900 -894 -888 -882 -876 -870 -864  -30  -24   -1    0    1   24   30
  864  870  876  882  888  894  900]
min offsets -900
max offsets 900
Data memory (bytes): 4378752
Offsets memory (bytes): 84
Total memory (bytes): 4378836
Total memory: 4.18 MB
displaying info for wang4
------------------------------
Shape: (26068, 26068)
Data shape: (23, 26068)
Offsets length: 23
[-900 -896 -892 -888 -884 -880 -876 -872  -30  -26   -1    0    1   26
   30  872  876  880  884  888  892  896  900]
min offsets -900
max offsets 900
Data memory (bytes): 4796512
Offsets memory (bytes): 92
Total memory (bytes): 4796604
Total memory: 4.57 MB
displaying info for s3dkt3m2
------------------------------


/tmp/ipykernel_863834/383998058.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 655 diagonals is inefficient
  A_dia = A.todia()


Shape: (90449, 90449)
Data shape: (655, 90449)
Offsets length: 655
[-614 -613 -612 -611 -610 -609 -608 -607 -606 -605 -604 -603 -602 -601
 -600 -599 -598 -597 -596 -595 -594 -593 -592 -591 -590 -589 -588 -587
 -586 -585 -584 -583 -582 -581 -580 -579 -578 -577 -576 -575 -574 -573
 -572 -571 -570 -569 -568 -567 -566 -565 -564 -563 -562 -561 -560 -559
 -558 -557 -556 -555 -554 -553 -552 -551 -550 -549 -548 -547 -546 -545
 -544 -543 -542 -541 -540 -539 -538 -537 -536 -535 -534 -533 -532 -531
 -530 -529 -528 -527 -526 -525 -524 -523 -522 -521 -520 -519 -518 -517
 -516 -515 -514 -513 -512 -511 -510 -509 -508 -507 -506 -505 -504 -503
 -502 -501 -500 -499 -498 -497 -496 -495 -494 -493 -492 -491 -490 -489
 -488 -487 -486 -485 -484 -483 -482 -481 -480 -479 -478 -477 -476 -475
 -474 -473 -472 -471 -470 -469 -468 -467 -466 -465 -464 -463 -462 -461
 -460 -459 -458 -457 -456 -455 -454 -453 -452 -451 -450 -449 -448 -447
 -446 -445 -444 -443 -442 -441 -440 -439 -438 -437 -436 -435 -434 -433
 -432 -431

/tmp/ipykernel_863834/383998058.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 661 diagonals is inefficient
  A_dia = A.todia()


Shape: (90449, 90449)
Data shape: (661, 90449)
Offsets length: 661
[-614 -613 -612 -611 -610 -609 -608 -607 -606 -605 -604 -603 -602 -601
 -600 -599 -598 -597 -596 -595 -594 -593 -592 -591 -590 -589 -588 -587
 -586 -585 -584 -583 -582 -581 -580 -579 -578 -577 -576 -575 -574 -573
 -572 -571 -570 -569 -568 -567 -566 -565 -564 -563 -562 -561 -560 -559
 -558 -557 -556 -555 -554 -553 -552 -551 -550 -549 -548 -547 -546 -545
 -544 -543 -542 -541 -540 -539 -538 -537 -536 -535 -534 -533 -532 -531
 -530 -529 -528 -527 -526 -525 -524 -523 -522 -521 -520 -519 -518 -517
 -516 -515 -514 -513 -512 -511 -510 -509 -508 -507 -506 -505 -504 -503
 -502 -501 -500 -499 -498 -497 -496 -495 -494 -493 -492 -491 -490 -489
 -488 -487 -486 -485 -484 -483 -482 -481 -480 -479 -478 -477 -476 -475
 -474 -473 -472 -471 -470 -469 -468 -467 -466 -465 -464 -463 -462 -461
 -460 -459 -458 -457 -456 -455 -454 -453 -452 -451 -450 -449 -448 -447
 -446 -445 -444 -443 -442 -441 -440 -439 -438 -437 -436 -435 -434 -433
 -432 -431

/tmp/ipykernel_863834/383998058.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 169 diagonals is inefficient
  A_dia = A.todia()
/tmp/ipykernel_863834/383998058.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 197 diagonals is inefficient
  A_dia = A.todia()


Shape: (9506, 9506)
Data shape: (197, 9506)
Offsets length: 197
[-98 -97 -96 -95 -94 -93 -92 -91 -90 -89 -88 -87 -86 -85 -84 -83 -82 -81
 -80 -79 -78 -77 -76 -75 -74 -73 -72 -71 -70 -69 -68 -67 -66 -65 -64 -63
 -62 -61 -60 -59 -58 -57 -56 -55 -54 -53 -52 -51 -50 -49 -48 -47 -46 -45
 -44 -43 -42 -41 -40 -39 -38 -37 -36 -35 -34 -33 -32 -31 -30 -29 -28 -27
 -26 -25 -24 -23 -22 -21 -20 -19 -18 -17 -16 -15 -14 -13 -12 -11 -10  -9
  -8  -7  -6  -5  -4  -3  -2  -1   0   1   2   3   4   5   6   7   8   9
  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27
  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45
  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63
  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81
  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98]
min offsets -98
max offsets 98
Data memory (bytes): 14981456
Offsets memory (bytes): 788
Total memory (bytes): 14982244
Total memory: 14

/tmp/ipykernel_863834/383998058.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 897 diagonals is inefficient
  A_dia = A.todia()


Shape: (503625, 503625)
Data shape: (897, 503625)
Offsets length: 897
[-859 -858 -857 -856 -855 -854 -853 -852 -851 -850 -849 -848 -847 -846
 -845 -844 -843 -842 -841 -840 -839 -838 -837 -836 -835 -834 -833 -832
 -831 -830 -829 -828 -827 -826 -825 -824 -823 -822 -821 -820 -819 -818
 -817 -816 -815 -814 -813 -812 -811 -810 -809 -808 -807 -806 -805 -804
 -803 -802 -801 -800 -799 -798 -797 -796 -795 -794 -793 -792 -791 -790
 -789 -788 -787 -786 -785 -784 -783 -782 -781 -780 -779 -778 -777 -776
 -775 -774 -773 -772 -771 -770 -769 -768 -767 -766 -765 -764 -763 -762
 -761 -760 -759 -758 -757 -756 -755 -754 -753 -752 -751 -750 -749 -748
 -747 -746 -745 -744 -743 -742 -741 -740 -739 -738 -737 -736 -735 -734
 -733 -732 -731 -730 -729 -728 -727 -726 -725 -724 -723 -722 -721 -720
 -719 -718 -717 -716 -715 -714 -713 -712 -711 -710 -709 -708 -707 -706
 -705 -704 -703 -702 -701 -700 -699 -698 -697 -696 -695 -694 -693 -692
 -691 -690 -689 -688 -687 -686 -685 -684 -683 -682 -681 -680 -679 -678
 -677 -

In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    row_indices, col_indices, lengths, values = encode_cdsf(A)
    print(row_indices.size)
    print(col_indices.size)
    print(lengths.size)
    print(values.size)

    cdsf_memory_report(row_indices, col_indices, lengths, values)

displaying info for wang3
------------------------------
1815
1815
1815
177168
Arrays:
  row_indices: 7.09 KB
  col_indices: 7.09 KB
  lengths    : 7.09 KB
  values     : 1384.12 KB
Total: 1405.39 KB  (1.37 MB)
displaying info for wang4
------------------------------
1819
1819
1819
177196
Arrays:
  row_indices: 7.11 KB
  col_indices: 7.11 KB
  lengths    : 7.11 KB
  values     : 1384.34 KB
Total: 1405.66 KB  (1.37 MB)
displaying info for s3dkt3m2
------------------------------
450993
450993
450993
3753461
Arrays:
  row_indices: 1761.69 KB
  col_indices: 1761.69 KB
  lengths    : 1761.69 KB
  values     : 29323.91 KB
Total: 34608.99 KB  (33.80 MB)
displaying info for s3dkq4m2
------------------------------
451293
451293
451293
4820891
Arrays:
  row_indices: 1762.86 KB
  col_indices: 1762.86 KB
  lengths    : 1762.86 KB
  values     : 37663.21 KB
Total: 42951.80 KB  (41.95 MB)
displaying info for kim1
------------------------------
4269
4269
4269
933195
Arrays:
  row_indices: 16.68 KB
  

/tmp/ipykernel_3021350/911313367.py:22: ComplexWarning: Casting complex values to real discards the imaginary part
  data = A.data.astype(np.float64, copy=False)


14807
14807
14807
11330020
Arrays:
  row_indices: 57.84 KB
  col_indices: 57.84 KB
  lengths    : 57.84 KB
  values     : 88515.78 KB
Total: 88689.30 KB  (86.61 MB)
displaying info for nemeth21
------------------------------
126289
126289
126289
1173746
Arrays:
  row_indices: 493.32 KB
  col_indices: 493.32 KB
  lengths    : 493.32 KB
  values     : 9169.89 KB
Total: 10649.84 KB  (10.40 MB)
displaying info for nemeth22
------------------------------
136339
136339
136339
1358832
Arrays:
  row_indices: 532.57 KB
  col_indices: 532.57 KB
  lengths    : 532.57 KB
  values     : 10615.88 KB
Total: 12213.60 KB  (11.93 MB)
displaying info for crystk02
------------------------------
152923
152923
152923
968583
Arrays:
  row_indices: 597.36 KB
  col_indices: 597.36 KB
  lengths    : 597.36 KB
  values     : 7567.05 KB
Total: 9359.12 KB  (9.14 MB)
displaying info for crystk03
------------------------------
274703
274703
274703
1751178
Arrays:
  row_indices: 1073.06 KB
  col_indices: 1073.06 KB
 

In [ ]:
A_dia.shape[1]

503625

In [ ]:
def time_spmv_dia(A_dia, reps=20, seed=42):
    n = A_dia.shape[1]
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n).astype(np.float64)

    # warm-up
    _ = A_dia @ x

    t0 = perf_counter()
    for _ in range(reps):
        y = A_dia @ x
    t_ms = (perf_counter() - t0) / reps * 1000.0  # ms per call

    print(f"Shape: {A_dia.shape}, diagonals: {len(A_dia.offsets)}")
    print(f"y = A_dia @ x  : {t_ms:.2f} ms per call")

    return t_ms


In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    A = coo_matrix(A)

    # Convert COO to DIA
    A_dia = A.todia()
    time_spmv_dia(A_dia, reps=10)

displaying info for wang3
------------------------------
Shape: (26064, 26064), diagonals: 21
y = A_dia @ x  : 0.34 ms per call
displaying info for wang4
------------------------------
Shape: (26068, 26068), diagonals: 23
y = A_dia @ x  : 0.37 ms per call
displaying info for s3dkt3m2
------------------------------


/tmp/ipykernel_3021350/966095400.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 655 diagonals is inefficient
  A_dia = A.todia()


Shape: (90449, 90449), diagonals: 655
y = A_dia @ x  : 67.76 ms per call
displaying info for s3dkq4m2
------------------------------


/tmp/ipykernel_3021350/966095400.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 661 diagonals is inefficient
  A_dia = A.todia()


Shape: (90449, 90449), diagonals: 661
y = A_dia @ x  : 68.43 ms per call
displaying info for kim1
------------------------------
Shape: (38415, 38415), diagonals: 25
y = A_dia @ x  : 2.42 ms per call
displaying info for kim2
------------------------------
Shape: (456976, 456976), diagonals: 25
y = A_dia @ x  : 38.55 ms per call
displaying info for nemeth21
------------------------------
Shape: (9506, 9506), diagonals: 169
y = A_dia @ x  : 1.08 ms per call
displaying info for nemeth22
------------------------------


/tmp/ipykernel_3021350/966095400.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 169 diagonals is inefficient
  A_dia = A.todia()
/tmp/ipykernel_3021350/966095400.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 197 diagonals is inefficient
  A_dia = A.todia()


Shape: (9506, 9506), diagonals: 197
y = A_dia @ x  : 1.32 ms per call
displaying info for crystk02
------------------------------
Shape: (13965, 13965), diagonals: 99
y = A_dia @ x  : 0.97 ms per call
displaying info for crystk03
------------------------------
Shape: (24696, 24696), diagonals: 99
y = A_dia @ x  : 1.81 ms per call
displaying info for af_1_k101
------------------------------


/tmp/ipykernel_3021350/966095400.py:10: SparseEfficiencyWarning: Constructing a DIA matrix with 897 diagonals is inefficient
  A_dia = A.todia()


Shape: (503625, 503625), diagonals: 897
y = A_dia @ x  : 550.72 ms per call
displaying info for af_2_k101
------------------------------
Shape: (503625, 503625), diagonals: 897
y = A_dia @ x  : 549.56 ms per call


In [ ]:
%pip install --upgrade --force-reinstall numba llvmlite

Defaulting to user installation because normal site-packages is not writeable
  Using cached numba-0.62.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.5 MB)
  Using cached llvmlite-0.45.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (56.3 MB)
  Using cached numpy-2.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.3
    Uninstalling numpy-2.3.3:
      Successfully uninstalled numpy-2.3.3
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.45.0
    Uninstalling llvmlite-0.45.0:
      Successfully uninstalled llvmlite-0.45.0
  Attempting uninstall: numba
    Found existing installation: numba 0.62.0
    Uninstalling numba-0.62.0:
      Successfully uninstalled numba-0.62.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

In [ ]:
@nb.njit(cache=True, fastmath=True)
def cdsf_spmv_numba(row_idx, col_idx, lengths, values, nrows, ncols, x):
    y = np.zeros(nrows, dtype=np.float64)
    pos = 0
    K = lengths.shape[0]
    for k in range(K):
        r0 = row_idx[k]
        c0 = col_idx[k]
        L  = lengths[k]
        for i in range(L):
            y[r0 + i] += values[pos + i] * x[c0 + i]
        pos += L
    return y

In [ ]:
# ---------- Benchmark helper ----------
def time_cdsf_spmv_numba(row_idx, col_idx, lengths, values, shape,reps=20, seed=42):
    """
    Times y = A @ x for CDSF (Numba kernel).
    Inputs:
      - row_idx, col_idx, lengths: int32 arrays
      - values: float64 array
      - shape: (nrows, ncols)
    Returns: ms per call (float)
    """
    nrows, ncols = shape
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(ncols).astype(np.float64)

    # --- Warm-up #1: trigger JIT compilation (excluded from timing)
    _ = cdsf_spmv_numba(row_idx, col_idx, lengths, values, nrows, ncols, x)

    # --- Warm-up #2: one steady-state run (also excluded)
    _ = cdsf_spmv_numba(row_idx, col_idx, lengths, values, nrows, ncols, x)

    # --- Timed runs
    t0 = perf_counter()
    for _ in range(reps):
        y = cdsf_spmv_numba(row_idx, col_idx, lengths, values, nrows, ncols, x)
    t_ms = (perf_counter() - t0) / reps * 1000.0

    print(f"CDSF SpMV (Numba): {t_ms:.2f} ms per call  |  shape={shape}, clusters={len(lengths)}, values={values.size}")
    return t_ms

In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    row_indices, col_indices, lengths, values = encode_cdsf(A)
    time_cdsf_spmv_numba(row_indices, col_indices, lengths, values, A.shape, reps=10)

displaying info for wang3
------------------------------
CDSF SpMV (Numba): 0.32 ms per call  |  shape=(26064, 26064), clusters=1815, values=177168
displaying info for wang4
------------------------------
CDSF SpMV (Numba): 0.33 ms per call  |  shape=(26068, 26068), clusters=1819, values=177196
displaying info for s3dkt3m2
------------------------------
CDSF SpMV (Numba): 7.89 ms per call  |  shape=(90449, 90449), clusters=450993, values=3753461
displaying info for s3dkq4m2
------------------------------
CDSF SpMV (Numba): 10.25 ms per call  |  shape=(90449, 90449), clusters=451293, values=4820891
displaying info for kim1
------------------------------
CDSF SpMV (Numba): 1.65 ms per call  |  shape=(38415, 38415), clusters=4269, values=933195
displaying info for kim2
------------------------------


/tmp/ipykernel_3021350/911313367.py:22: ComplexWarning: Casting complex values to real discards the imaginary part
  data = A.data.astype(np.float64, copy=False)


CDSF SpMV (Numba): 21.55 ms per call  |  shape=(456976, 456976), clusters=14807, values=11330020
displaying info for nemeth21
------------------------------
CDSF SpMV (Numba): 2.30 ms per call  |  shape=(9506, 9506), clusters=126289, values=1173746
displaying info for nemeth22
------------------------------
CDSF SpMV (Numba): 2.65 ms per call  |  shape=(9506, 9506), clusters=136339, values=1358832
displaying info for crystk02
------------------------------
CDSF SpMV (Numba): 1.96 ms per call  |  shape=(13965, 13965), clusters=152923, values=968583
displaying info for crystk03
------------------------------
CDSF SpMV (Numba): 3.79 ms per call  |  shape=(24696, 24696), clusters=274703, values=1751178
displaying info for af_1_k101
------------------------------
CDSF SpMV (Numba): 43.14 ms per call  |  shape=(503625, 503625), clusters=2616475, values=17550675
displaying info for af_2_k101
------------------------------
CDSF SpMV (Numba): 42.62 ms per call  |  shape=(503625, 503625), cluste

In [ ]:
%pip install -q memory_profiler

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
from time import perf_counter
from scipy.sparse import coo_matrix
from memory_profiler import memory_usage

In [ ]:
def _coo_to_dia(A_coo):
    # Function wrapper so memory_profiler can run it
    return A_coo.todia()

In [ ]:
def time_and_peak_mem_coo_to_dia(A_coo, reps=5, interval=0.001):
    """
    Measures average time (ms) and peak memory (MB) for A_coo.todia().
    Uses memory_profiler to measure peak RSS per run, then reports the max peak.
    """
    times_ms = []
    peak_deltas_mb = []
    A_dia = None

    # One warm-up (not timed) to avoid cold-start noise
    _ = A_coo.todia()

    for _ in range(reps):
        t0 = perf_counter()
        mem_trace, A_dia = memory_usage(
            (_coo_to_dia, (A_coo,), {}), retval=True, interval=interval
        )
        elapsed_ms = (perf_counter() - t0) * 1000.0
        # memory_usage returns a list of RSS (MB); take peak - start as delta for the run
        peak_delta_mb = max(mem_trace) - mem_trace[0]

        times_ms.append(elapsed_ms)
        peak_deltas_mb.append(peak_delta_mb)

    avg_ms = float(np.mean(times_ms))
    peak_mb = float(max(peak_deltas_mb))  # the worst (highest) peak across runs

    print(f"COO->DIA: {avg_ms:.2f} ms per call (avg over {reps})")
    print(f"Peak memory during conversion: {peak_mb:.2f} MB")
    return A_dia, avg_ms, peak_mb

In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    A_coo= A.tocoo()
    A_dia, t_ms, peak_mb = time_and_peak_mem_coo_to_dia(A_coo, reps=5)

displaying info for wang3
------------------------------


TypeError: 'float' object is not iterable

In [ ]:
# --- Benchmark COO -> CDSF ---
def time_and_peak_mem_coo_to_cdsf(A_coo, reps=5, interval=0.0001):
    """
    Measures average time (ms) and peak memory (KB) for COO->CDSF encoding.
    Uses memory_profiler to capture peak RSS delta.
    """
    times_ms = []
    peak_deltas_kb = []
    arrays = None

    # Warm-up
    _ = encode_cdsf(A_coo)

    for run in range(reps):
        t0 = perf_counter()
        mem_trace, arrays = memory_usage(
            (encode_cdsf, (A_coo,), {}), retval=True, interval=interval
        )
        elapsed_ms = (perf_counter() - t0) * 1000.0
        peak_delta_kb = (max(mem_trace) - mem_trace[0]) * 1024

        print(f"Run {run+1}: start={mem_trace[0]*1024:.2f} MB, peak={max(mem_trace)*1024:.2f} MB, delta={peak_delta_kb:.2f} KB, time={elapsed_ms:.2f} ms")

        times_ms.append(elapsed_ms)
        peak_deltas_kb.append(peak_delta_kb)

    avg_ms = float(np.mean(times_ms))
    peak_kb = float(max(peak_deltas_kb))

    print("\n=== Summary COO->CDSF ===")
    print(f"Average time: {avg_ms:.2f} ms per call (over {reps})")
    print(f"Peak memory during conversion: {peak_kb:.2f} KB")

    return arrays, avg_ms, peak_kb

In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    time_and_peak_mem_coo_to_cdsf(A, reps=10, interval=0.001)

displaying info for wang3
------------------------------
Run 1: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=46.92 ms
Run 2: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=52.34 ms
Run 3: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=45.69 ms
Run 4: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=47.01 ms
Run 5: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=45.56 ms
Run 6: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=56.24 ms
Run 7: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=53.40 ms
Run 8: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=38.74 ms
Run 9: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=38.63 ms
Run 10: start=1372552.00 MB, peak=1372552.00 MB, delta=0.00 KB, time=36.79 ms

=== Summary COO->CDSF ===
Average time: 46.13 ms per call (over 10)
Peak memory during conversion: 0.00 KB
displaying info for wang4
------------------------------
Run 1: 

/tmp/ipykernel_433962/911313367.py:22: ComplexWarning: Casting complex values to real discards the imaginary part
  data = A.data.astype(np.float64, copy=False)


Run 1: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=146.03 ms
Run 2: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=157.34 ms
Run 3: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=148.69 ms
Run 4: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=130.05 ms
Run 5: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=135.93 ms
Run 6: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=146.00 ms
Run 7: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=138.48 ms
Run 8: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=141.76 ms
Run 9: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=139.61 ms
Run 10: start=1373192.00 MB, peak=1373192.00 MB, delta=0.00 KB, time=143.40 ms

=== Summary COO->CDSF ===
Average time: 142.73 ms per call (over 10)
Peak memory during conversion: 0.00 KB
displaying info for kim2
------------------------------
Run 1: start=1722264.00 MB, peak=1899008.00 MB, delta=

In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    A = coo_matrix(A)

    # Convert COO to DIA
    A_csr = A.tocsr()

    data_bytes    = A_csr.data.nbytes
    indices_bytes = A_csr.indices.nbytes
    indptr_bytes  = A_csr.indptr.nbytes
    total_bytes   = data_bytes + indices_bytes + indptr_bytes

    print(f"Shape: {A_csr.shape}, nnz={A_csr.nnz}")
    print(f"  data   : {data_bytes/1024:.2f} KB")
    print(f"  indices: {indices_bytes/1024:.2f} KB")
    print(f"  indptr : {indptr_bytes/1024:.2f} KB")
    print(f"Total memory: {total_bytes/1024:.2f} KB  ({total_bytes/(1024*1024):.2f} MB)")

displaying info for wang3
------------------------------
Shape: (26064, 26064), nnz=177168
  data   : 1384.12 KB
  indices: 692.06 KB
  indptr : 101.82 KB
Total memory: 2178.00 KB  (2.13 MB)
displaying info for wang4
------------------------------
Shape: (26068, 26068), nnz=177196
  data   : 1384.34 KB
  indices: 692.17 KB
  indptr : 101.83 KB
Total memory: 2178.35 KB  (2.13 MB)
displaying info for s3dkt3m2
------------------------------
Shape: (90449, 90449), nnz=3753461
  data   : 29323.91 KB
  indices: 14661.96 KB
  indptr : 353.32 KB
Total memory: 44339.19 KB  (43.30 MB)
displaying info for s3dkq4m2
------------------------------
Shape: (90449, 90449), nnz=4820891
  data   : 37663.21 KB
  indices: 18831.61 KB
  indptr : 353.32 KB
Total memory: 56848.14 KB  (55.52 MB)
displaying info for kim1
------------------------------
Shape: (38415, 38415), nnz=933195
  data   : 14581.17 KB
  indices: 3645.29 KB
  indptr : 150.06 KB
Total memory: 18376.53 KB  (17.95 MB)
displaying info for kim2

In [ ]:
def time_spmv_csr(A_csr, reps=20, seed=42):
    n = A_csr.shape[1]
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n).astype(np.float64)

    # warm-up
    _ = A_csr @ x

    t0 = perf_counter()
    for _ in range(reps):
        y = A_csr @ x
    t_ms = (perf_counter() - t0) / reps * 1000.0  # ms per call

    print(f"y = A_csr @ x  : {t_ms:.2f} ms per call")

    return t_ms

In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    A = coo_matrix(A)

    # Convert COO to DIA
    A_csr = A.tocsr()
    time_spmv_csr(A_csr, reps=10)

displaying info for wang3
------------------------------
y = A_csr @ x  : 0.21 ms per call
displaying info for wang4
------------------------------
y = A_csr @ x  : 0.35 ms per call
displaying info for s3dkt3m2
------------------------------
y = A_csr @ x  : 4.64 ms per call
displaying info for s3dkq4m2
------------------------------
y = A_csr @ x  : 5.69 ms per call
displaying info for kim1
------------------------------
y = A_csr @ x  : 2.10 ms per call
displaying info for kim2
------------------------------
y = A_csr @ x  : 28.32 ms per call
displaying info for nemeth21
------------------------------
y = A_csr @ x  : 1.48 ms per call
displaying info for nemeth22
------------------------------
y = A_csr @ x  : 1.80 ms per call
displaying info for crystk02
------------------------------
y = A_csr @ x  : 1.19 ms per call
displaying info for crystk03
------------------------------
y = A_csr @ x  : 2.11 ms per call
displaying info for af_1_k101
------------------------------
y = A_csr @ 

In [ ]:
def _coo_to_csr(A_csr):
    # Function wrapper so memory_profiler can run it
    return A_coo.tocsr()

In [ ]:
def time_and_peak_mem_coo_to_csr(A_coo, reps=5, interval=0.001):
    """
    Measures average time (ms) and peak memory (MB) for A_coo.todia().
    Uses memory_profiler to measure peak RSS per run, then reports the max peak.
    """
    times_ms = []
    peak_deltas_mb = []
    A_csr = None

    # One warm-up (not timed) to avoid cold-start noise
    _ = A_coo.tocsr()

    for _ in range(reps):
        t0 = perf_counter()
        mem_trace, A_csr = memory_usage(
            (_coo_to_csr, (A_coo,), {}), retval=True, interval=interval
        )
        elapsed_ms = (perf_counter() - t0) * 1000.0
        # memory_usage returns a list of RSS (MB); take peak - start as delta for the run
        peak_delta_mb = max(mem_trace) - mem_trace[0]

        times_ms.append(elapsed_ms)
        peak_deltas_mb.append(peak_delta_mb)

    avg_ms = float(np.mean(times_ms))
    peak_mb = float(max(peak_deltas_mb))  # the worst (highest) peak across runs

    print(f"COO->CSR: {avg_ms:.2f} ms per call (avg over {reps})")
    print(f"Peak memory during conversion: {peak_mb:.2f} MB")
    return A_csr, avg_ms, peak_mb

In [ ]:
for matrix in matrix_list:
    print(f"displaying info for {matrix}")
    print("------------------------------")

    # Load the matrix from an .mtx file
    A = mmread(matrix + ".mtx")
    A_coo = A.tocoo()
    A_csr, t_ms, peak_mb = time_and_peak_mem_coo_to_csr(A_coo, reps=5)

displaying info for wang3
------------------------------
COO->CSR: 28.94 ms per call (avg over 5)
Peak memory during conversion: 0.00 MB
displaying info for wang4
------------------------------
COO->CSR: 35.08 ms per call (avg over 5)
Peak memory during conversion: 0.00 MB
displaying info for s3dkt3m2
------------------------------
COO->CSR: 95.75 ms per call (avg over 5)
Peak memory during conversion: 0.00 MB
displaying info for s3dkq4m2
------------------------------
COO->CSR: 112.70 ms per call (avg over 5)
Peak memory during conversion: 0.00 MB
displaying info for kim1
------------------------------
COO->CSR: 57.76 ms per call (avg over 5)
Peak memory during conversion: 0.00 MB
displaying info for kim2
------------------------------
COO->CSR: 236.31 ms per call (avg over 5)
Peak memory during conversion: 172.93 MB
displaying info for nemeth21
------------------------------
COO->CSR: 55.79 ms per call (avg over 5)
Peak memory during conversion: 0.00 MB
displaying info for nemeth22
-

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix
from typing import Optional, Sequence, Tuple, Iterable

# -----------------------------
# Generator (same as before)
# -----------------------------
def create_banded_sparse_matrix_irregular(
    n: int,
    offsets: Iterable[int],
    density: float = 1.0,
    per_diag_density: Optional[Sequence[float]] = None,
    per_diag_trim: Optional[Sequence[Tuple[int,int]]] = None,
    dtype=np.float64,
    seed: Optional[int] = None,
) -> coo_matrix:
    rng = np.random.default_rng(seed)
    offsets = np.array(list(offsets), dtype=int)
    k = offsets.size

    if per_diag_density is not None:
        diag_ps = np.asarray(per_diag_density, dtype=float)
        if diag_ps.shape != (k,):
            raise ValueError(f"per_diag_density must have shape ({k},), got {diag_ps.shape}")
    else:
        diag_ps = np.full(k, float(density), dtype=float)

    if per_diag_trim is not None:
        if len(per_diag_trim) != k:
            raise ValueError(f"per_diag_trim must have length {k}, got {len(per_diag_trim)}")
        trims = np.array(per_diag_trim, dtype=int)
        if (trims < 0).any():
            raise ValueError("per_diag_trim entries must be non-negative")
    else:
        trims = np.zeros((k, 2), dtype=int)

    row_chunks, col_chunks, val_chunks = [], [], []

    for idx, (off, p) in enumerate(zip(offsets, diag_ps)):
        if p <= 0.0:
            continue

        L_full = n - abs(off)
        if L_full <= 0:
            continue

        if off >= 0:
            r0, c0 = 0, off
        else:
            r0, c0 = -off, 0

        head, tail = trims[idx]
        L_eff = L_full - head - tail
        if L_eff <= 0:
            continue

        r_start = r0 + head
        c_start = c0 + head

        mask = rng.random(L_eff) < p
        k_keep = int(mask.sum())
        if k_keep == 0:
            continue

        rows = (np.arange(r_start, r_start + L_eff, dtype=np.int32))[mask]
        cols = (np.arange(c_start, c_start + L_eff, dtype=np.int32))[mask]
        vals = rng.random(k_keep, dtype=dtype)

        row_chunks.append(rows)
        col_chunks.append(cols)
        val_chunks.append(vals)

    if not row_chunks:
        return coo_matrix((n, n), dtype=dtype)

    row = np.concatenate(row_chunks, dtype=np.int32)
    col = np.concatenate(col_chunks, dtype=np.int32)
    dat = np.concatenate(val_chunks, dtype=dtype)
    return coo_matrix((dat, (row, col)), shape=(n, n), dtype=dtype)

# -----------------------------
# Offset sampler: size ∈ [500,1000]
# -----------------------------
def make_large_offset_set(rng: np.random.Generator, n: int,
                          min_size: int = 500,
                          max_size: int = 1000) -> np.ndarray:
    """
    Pick an offset set of size between min_size and max_size.
    Offsets sampled from full range [-n+1, n-1].
    """
    all_offs = np.arange(-(n-1), n, dtype=int)
    k = int(rng.integers(min_size, max_size + 1))  # number of diagonals
    k = min(k, all_offs.size)
    offs = rng.choice(all_offs, size=k, replace=False)
    return np.unique(offs)

# -----------------------------
# Example: build 3 matrices
# -----------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(42)
    sizes = [10000,40000,70000,100000]  # smaller for demo, must be >= 1000 to allow 1000 diagonals

    matrices = []
    for n in sizes:
        offsets = make_large_offset_set(rng, n, 500, 1000)
        per_diag_density = 0.7 * np.ones(len(offsets))  # same density
        A = create_banded_sparse_matrix_irregular(
            n=n,
            offsets=offsets,
            per_diag_density=per_diag_density,
            dtype=np.float64,
            seed=int(rng.integers(0, 1_000_000))
        )
        print(f"n={n} | #offsets={len(offsets)} | nnz={A.nnz:,}")
        matrices.append(A)




dia = []
for ino in matrices:
    dia.append(ino.todia())
    print("end")

n=10000 | #offsets=544 | nnz=1,907,663
n=40000 | #offsets=715 | nnz=10,111,607
n=70000 | #offsets=671 | nnz=16,570,031
n=100000 | #offsets=542 | nnz=19,006,922


/tmp/ipykernel_1347340/3519631059.py:123: SparseEfficiencyWarning: Constructing a DIA matrix with 544 diagonals is inefficient
  dia.append(ino.todia())


end


/tmp/ipykernel_1347340/3519631059.py:123: SparseEfficiencyWarning: Constructing a DIA matrix with 715 diagonals is inefficient
  dia.append(ino.todia())


end


/tmp/ipykernel_1347340/3519631059.py:123: SparseEfficiencyWarning: Constructing a DIA matrix with 671 diagonals is inefficient
  dia.append(ino.todia())


end


/tmp/ipykernel_1347340/3519631059.py:123: SparseEfficiencyWarning: Constructing a DIA matrix with 542 diagonals is inefficient
  dia.append(ino.todia())


end


In [ ]:
for matrix in dia:




    # Memory usage
    data_bytes = matrix.data.nbytes        # storage for diagonal values
    offset_bytes = matrix.offsets.nbytes   # storage for offsets
    total_bytes = data_bytes + offset_bytes

    print("Shape:", matrix.shape)
    print("Data shape:", matrix.data.shape)   # (num_diagonals, n)
    print("Offsets length:", len(matrix.offsets))
    print("Data memory (bytes):", data_bytes)
    print("Offsets memory (bytes):", offset_bytes)
    print("Total memory (bytes):", total_bytes)
    print(f"Total memory: {total_bytes/(1024*1024):.2f} MB")
    print("-----------------------------------------------")

Shape: (10000, 10000)
Data shape: (544, 10000)
Offsets length: 544
Data memory (bytes): 43520000
Offsets memory (bytes): 2176
Total memory (bytes): 43522176
Total memory: 41.51 MB
-----------------------------------------------
Shape: (40000, 40000)
Data shape: (715, 40000)
Offsets length: 715
Data memory (bytes): 228800000
Offsets memory (bytes): 2860
Total memory (bytes): 228802860
Total memory: 218.20 MB
-----------------------------------------------
Shape: (70000, 70000)
Data shape: (671, 70000)
Offsets length: 671
Data memory (bytes): 375760000
Offsets memory (bytes): 2684
Total memory (bytes): 375762684
Total memory: 358.36 MB
-----------------------------------------------
Shape: (100000, 100000)
Data shape: (542, 100000)
Offsets length: 542
Data memory (bytes): 433600000
Offsets memory (bytes): 2168
Total memory (bytes): 433602168
Total memory: 413.52 MB
-----------------------------------------------


In [ ]:
for matrix in matrices:



    row_indices, col_indices, lengths, values = encode_cdsf(matrix)
    print(row_indices.size)
    print(col_indices.size)
    print(lengths.size)
    print(values.size)

    cdsf_memory_report(row_indices, col_indices, lengths, values)

572553
572553
572553
1907663
Arrays:
  row_indices: 2236.54 KB
  col_indices: 2236.54 KB
  lengths    : 2236.54 KB
  values     : 14903.62 KB
Total: 21613.22 KB  (21.11 MB)
3034907
3034907
3034907
10111607
Arrays:
  row_indices: 11855.11 KB
  col_indices: 11855.11 KB
  lengths    : 11855.11 KB
  values     : 78996.93 KB
Total: 114562.25 KB  (111.88 MB)
4972094
4972094
4972094
16570031
Arrays:
  row_indices: 19422.24 KB
  col_indices: 19422.24 KB
  lengths    : 19422.24 KB
  values     : 129453.37 KB
Total: 187720.09 KB  (183.32 MB)
5700713
5700713
5700713
19006922
Arrays:
  row_indices: 22268.41 KB
  col_indices: 22268.41 KB
  lengths    : 22268.41 KB
  values     : 148491.58 KB
Total: 215296.81 KB  (210.25 MB)


In [ ]:
for matrix in dia:


    # Convert COO to DIA

    time_spmv_dia(matrix, reps=10)

Shape: (10000, 10000), diagonals: 544
y = A_dia @ x  : 2.55 ms per call
Shape: (40000, 40000), diagonals: 715
y = A_dia @ x  : 12.60 ms per call
Shape: (70000, 70000), diagonals: 671
y = A_dia @ x  : 23.83 ms per call
Shape: (100000, 100000), diagonals: 542
y = A_dia @ x  : 29.65 ms per call


In [ ]:
for matrix in matrices:


    row_indices, col_indices, lengths, values = encode_cdsf(matrix)
    time_cdsf_spmv_numba(row_indices, col_indices, lengths, values, matrix.shape, reps=10)

CDSF SpMV (Numba): 11.75 ms per call  |  shape=(10000, 10000), clusters=572553, values=1907663
CDSF SpMV (Numba): 61.67 ms per call  |  shape=(40000, 40000), clusters=3034907, values=10111607
CDSF SpMV (Numba): 100.03 ms per call  |  shape=(70000, 70000), clusters=4972094, values=16570031
CDSF SpMV (Numba): 114.04 ms per call  |  shape=(100000, 100000), clusters=5700713, values=19006922


In [ ]:
print("hello")

hello


In [ ]:
python3 --version

NameError: name 'python3' is not defined

In [ ]:
import scipy as np
print(np.__version__)

1.15.2


In [ ]:
import numba as nb
print(nb.__version__)

0.62.0


In [ ]:
pip show memory_profiler

Name: memory-profiler
Version: 0.61.0
Summary: A module for monitoring memory usage of a python program
Home-page: https://github.com/pythonprofilers/memory_profiler
Author: Fabian Pedregosa
Author-email: f@bianp.net
License: BSD
Location: /home/xim/.local/lib/python3.11/site-packages
Requires: psutil
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
CDSF vs CSR PageRank benchmark (Numba-accelerated CDSF matvec)

Save as: cdsf_vs_csr_final.py
Run: in a Python environment with numpy, scipy, and numba (recommended).

Produces CSV: /mnt/data/CDSF_vs_CSR_times_final.csv
"""

import time
import numpy as np
import scipy.sparse as sp
import pandas as pd
import math
import sys

# ----------------- User-tunable parameters -----------------
n = 10000                 # matrix dimension (increase to stress memory)
num_clusters = 6          # number of diagonal clusters
cluster_width = 6         # adjacent diagonals per cluster
density_per_diagonal = 0.02  # fraction of positions on each diagonal that are nonzero
iterations = 500         # number of PageRank iterations to run
alpha = 0.85             # PageRank damping factor
rng_seed = 42
use_numba = True         # try to use numba if available
# ----------------------------------------------------------

rng = np.random.default_rng(rng_seed)


def build_clustered_coo(n, num_clusters, cluster_width, density, rng):
    rows = []
    cols = []
    data = []

    # choose cluster centers (offsets) spread out
    spread = max(10, int(n / 200))  # adjust spread relative to n
    cluster_centers = np.linspace(-spread, spread, num_clusters).astype(int)

    for center in cluster_centers:
        for k in range(cluster_width):
            offset = center - (cluster_width // 2) + k
            i_min = max(0, -offset)
            i_max = min(n, n - offset)
            length = i_max - i_min
            if length <= 0:
                continue
            mask = rng.random(length) < density
            if not mask.any():
                # ensure at least one entry
                mask[rng.integers(0, length)] = True
            idxs = np.nonzero(mask)[0] + i_min
            jdxs = idxs + offset
            rows.append(idxs)
            cols.append(jdxs)
            data.append(rng.random(len(idxs)) + 0.1)

    if len(rows) == 0:
        raise RuntimeError("No nonzeros generated; adjust parameters.")

    rows = np.concatenate(rows).astype(np.int32)
    cols = np.concatenate(cols).astype(np.int32)
    data = np.concatenate(data).astype(np.float64)

    A_coo = sp.coo_matrix((data, (rows, cols)), shape=(n, n))
    return A_coo


def normalize_columns_to_stochastic(A_coo):
    # convert matrix so columns sum to 1 (column-stochastic), handle dangling cols
    col_sums = np.array(A_coo.sum(axis=0)).ravel()
    col_sums_safe = np.where(col_sums == 0, 1.0, col_sums)
    data = A_coo.data * (1.0 / col_sums_safe[A_coo.col])
    return sp.coo_matrix((data, (A_coo.row, A_coo.col)), shape=A_coo.shape)


def coo_to_cdsf_simple(coo):
    """
    Convert COO to CDSF-like representation:
      offsets: 1D array length m (unique offsets)
      data: shape (n, m) where column k contains diagonal values for offsets[k]
    This stores only diagonals that actually exist (no extra offsets).
    """
    r = coo.row
    c = coo.col
    v = coo.data
    offsets = np.unique(c - r)
    offsets.sort()
    m = len(offsets)
    data = np.zeros((coo.shape[0], m), dtype=np.float64)
    # build mapping offset -> column index in data
    offset_to_idx = {off: idx for idx, off in enumerate(offsets)}
    for rr, cc, vv in zip(r, c, v):
        idx = offset_to_idx[cc - rr]
        data[rr, idx] = vv
    return offsets.astype(np.int32), data


# Try to import numba and create a njit matvec
numba_available = False
if use_numba:
    try:
        import numba as nb
        numba_available = True
    except Exception:
        numba_available = False

if numba_available:
    # Numba-friendly CDSF matvec: loops over offsets and rows
    @nb.njit(fastmath=True, nogil=True, cache=True)
    def pr_matvec_cdsf_numba(offsets, data, x, y_out):
        """
        offsets: int32[:] length m
        data: float64[:, :] shape (n, m)
        x: float64[:] length n
        y_out: preallocated float64[:] length n (will be zeroed by caller or overwritten additively)
        Performs y_out += data[:, k] * x_shift for each offset
        """
        n_local = data.shape[0]
        m_local = data.shape[1]
        # zero y_out (caller may prefer zeroing outside)
        for i in range(n_local):
            y_out[i] = 0.0
        for k in range(m_local):
            off = offsets[k]
            if off == 0:
                # full length
                for i in range(n_local):
                    a = data[i, k]
                    if a != 0.0:
                        y_out[i] += a * x[i]
            elif off > 0:
                # j = i + off  => for i in 0..n-off-1, j in off..n-1
                limit = n_local - off
                for i in range(limit):
                    a = data[i, k]
                    if a != 0.0:
                        y_out[i] += a * x[i + off]
            else:
                # off < 0 : j = i + off  => i in -off .. n-1
                o = -off
                for i in range(o, n_local):
                    a = data[i, k]
                    if a != 0.0:
                        y_out[i] += a * x[i - o]

    def pr_matvec_cdsf_factory(offsets, data):
        # return a wrapper that allocates y and calls numba function
        def matvec(x):
            y = np.zeros_like(x)
            pr_matvec_cdsf_numba(offsets, data, x, y)
            return y
        return matvec
else:
    # fallback Python implementation (vectorized with slices)
    def pr_matvec_cdsf_py(offsets, data, x):
        n_local = data.shape[0]
        y = np.zeros_like(x)
        for k, off in enumerate(offsets):
            col = data[:, k]
            if off == 0:
                y += col * x
            elif off > 0:
                limit = n_local - off
                if limit > 0:
                    y[:limit] += col[:limit] * x[off:]
            else:
                o = -off
                if n_local - o > 0:
                    y[o:] += col[o:] * x[:n_local - o]
        return y

    def pr_matvec_cdsf_factory(offsets, data):
        return lambda x: pr_matvec_cdsf_py(offsets, data, x)


# -------------------- Build matrix and convert --------------------
print("Building clustered COO matrix (n={} ... )".format(n))
t0 = time.perf_counter()
A_coo = build_clustered_coo(n, num_clusters, cluster_width, density_per_diagonal, rng)
A_coo = normalize_columns_to_stochastic(A_coo)
t_build = time.perf_counter() - t0
print(f"Built COO in {t_build:.4f} s; nnz = {A_coo.nnz}")

# COO -> CSR
t0 = time.perf_counter()
A_csr = A_coo.tocsr()
t_csr_conv = time.perf_counter() - t0
print(f"COO -> CSR conversion: {t_csr_conv:.6f} s")

# COO -> CDSF (simple diag-columns)
t0 = time.perf_counter()
offsets, data = coo_to_cdsf_simple(A_coo)
t_cdsf_conv = time.perf_counter() - t0
print(f"COO -> CDSF conversion: {t_cdsf_conv:.6f} s")
print(f"CDSF: {len(offsets)} diagonals stored (data shape = {data.shape})")

# create CDSF matvec (numba-accelerated if available)
pr_cdsf = pr_matvec_cdsf_factory(offsets, data)

# Warm-up compilation if numba used
if numba_available:
    print("Warming up Numba-compiled function (first call will include compilation)...")
    xw = np.ones(n, dtype=np.float64) / n
    _ = pr_cdsf(xw)  # first call triggers compilation; measured later separately

# -------------------- PageRank iterations (CSR) --------------------
def pagerank_csr(Acsr, alpha, iterations, x0=None):
    nloc = Acsr.shape[0]
    if x0 is None:
        x = np.ones(nloc, dtype=np.float64) / nloc
    else:
        x = x0.copy()
    teleport = (1.0 - alpha) / nloc
    tstart = time.perf_counter()
    for _ in range(iterations):
        x = alpha * (Acsr.dot(x)) + teleport
    tend = time.perf_counter()
    return x, tend - tstart

# -------------------- PageRank iterations (CDSF) --------------------
def pagerank_cdsf(pr_cdsf_func, alpha, iterations, x0=None):
    nloc = data.shape[0]
    if x0 is None:
        x = np.ones(nloc, dtype=np.float64) / nloc
    else:
        x = x0.copy()
    teleport = (1.0 - alpha) / nloc
    tstart = time.perf_counter()
    for _ in range(iterations):
        y = pr_cdsf_func(x)
        x = alpha * y + teleport
    tend = time.perf_counter()
    return x, tend - tstart


# Run and time
print("Running PageRank with CSR ({} iterations)...".format(iterations))
x0 = np.ones(n, dtype=np.float64) / n
x_csr, t_csr_iter = pagerank_csr(A_csr, alpha, iterations, x0=x0)
print("CSR iterations time: {:.6f} s ({:.9f} s/iter)".format(t_csr_iter, t_csr_iter / iterations))

print("Running PageRank with CDSF ({} iterations)...".format(iterations))
x_cdsf, t_cdsf_iter = pagerank_cdsf(pr_cdsf, alpha, iterations, x0=x0)
print("CDSF iterations time: {:.6f} s ({:.9f} s/iter)".format(t_cdsf_iter, t_cdsf_iter / iterations))

# correctness check
diff_norm = np.linalg.norm(x_csr - x_cdsf)
print("L2 norm of difference between final PageRank vectors: {:.6e}".format(diff_norm))

# breakeven iteration calculation:
# want iters_b such that t_cdsf_conv + iters_b * t_cdsf_iter/iters < t_csr_conv + iters_b * t_csr_iter/iters
per_iter_csr = t_csr_iter / iterations
per_iter_cdsf = t_cdsf_iter / iterations

if per_iter_cdsf < per_iter_csr:
    # solve for iters_b
    iters_b = math.ceil((t_cdsf_conv - t_csr_conv) / (per_iter_csr - per_iter_cdsf))
    iters_b = max(0, iters_b)
else:
    iters_b = None  # CDSF doesn't reach breakeven in this configuration

# Summarize
summary = {
    "n": n,
    "nnz": int(A_coo.nnz),
    "num_offsets": int(len(offsets)),
    "csr_conv_s": t_csr_conv,
    "cdsf_conv_s": t_cdsf_conv,
    "csr_total_iters_s": t_csr_iter,
    "cdsf_total_iters_s": t_cdsf_iter,
    "csr_per_iter_s": per_iter_csr,
    "cdsf_per_iter_s": per_iter_cdsf,
    "breakeven_iters": iters_b,
    "numba_available": bool(numba_available),
    "time_build_coo_s": t_build
}

df = pd.DataFrame(list(summary.items()), columns=["metric", "value"])
csv_path = "/mnt/data/CDSF_vs_CSR_times_final.csv"
df.to_csv(csv_path, index=False)

print("\nSummary (also saved as CSV):")
print(df.to_string(index=False))
print(f"\nCSV saved to: {csv_path}")

if iters_b is None:
    print("\nCDSF did not reach breakeven under these parameters (per-iteration slower than CSR).")
else:
    print(f"\nEstimated breakeven iterations where CDSF becomes cheaper overall: {iters_b} iterations")
    print("That is, after", iters_b, "iterations the total cost (conv+iters*per_iter) for CDSF < CSR.")

# End of script


Building clustered COO matrix (n=10000 ... )
Built COO in 0.0040 s; nnz = 7080
COO -> CSR conversion: 0.000277 s
COO -> CDSF conversion: 0.004344 s
CDSF: 36 diagonals stored (data shape = (10000, 36))
Warming up Numba-compiled function (first call will include compilation)...
Running PageRank with CSR (500 iterations)...
CSR iterations time: 0.038649 s (0.000077297 s/iter)
Running PageRank with CDSF (500 iterations)...
CDSF iterations time: 0.392726 s (0.000785452 s/iter)
L2 norm of difference between final PageRank vectors: 1.424628e-19


OSError: Cannot save file into a non-existent directory: '/mnt/data'

In [ ]:
# preferred (ensures install into current kernel)
%pip install numpy scipy pandas numba requests tqdm




Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
CDSF vs CSR long-run benchmark (Numba-optimized CDSF)
- Build a large clustered-diagonal matrix (COO)
- Convert to CSR (SciPy) and to CDSF (cluster-block dense layout)
- Numba-jitted CDSF matvec (fast)
- Run many PageRank iterations and show breakeven point

Tune the parameters below for your machine.
"""

import time, math
import numpy as np
import scipy.sparse as sp
import pandas as pd

# ------------------ TUNABLE PARAMETERS (adjust to your hardware) ------------------
n = 40000                      # matrix dimension (reduce if you have low RAM)
num_clusters = 12              # number of diagonal clusters
cluster_width = 24             # number of adjacent diagonals in each cluster (wider => more dense blocks)
density_per_diagonal = 0.6     # fraction of positions on each diagonal filled (0..1). High => more local density
iterations = 2000              # number of PageRank / matvec iterations (large to amortize conversion)
alpha = 0.85                   # PageRank damping, not critical
rows_block_size = 4096         # block height for CDSF blocks (tune for cache)
rng_seed = 1234
use_numba = True               # must be True to get CDSF advantage
# ----------------------------------------------------------------------------------

rng = np.random.default_rng(rng_seed)

def build_clustered_coo(n, num_clusters, cluster_width, density, rng):
    """Build COO with clusters of adjacent diagonals; returns scipy.sparse.coo_matrix"""
    rows = []
    cols = []
    data = []
    # spread of cluster centers in offsets; make them around main diagonal
    spread = max(20, int(n / 200))
    centers = np.linspace(-spread, spread, num_clusters).astype(int)
    for center in centers:
        for k in range(cluster_width):
            offset = center - (cluster_width // 2) + k
            i_min = max(0, -offset)
            i_max = min(n, n - offset)
            length = i_max - i_min
            if length <= 0:
                continue
            # pick many entries per diagonal according to density
            mask = rng.random(length) < density
            # ensure at least one if density very small
            if not mask.any():
                mask[rng.integers(0, length)] = True
            idxs = np.nonzero(mask)[0] + i_min
            jdxs = idxs + offset
            rows.append(idxs)
            cols.append(jdxs)
            data.append(rng.random(len(idxs)) + 0.1)  # random positive weights
    rows = np.concatenate(rows).astype(np.int32)
    cols = np.concatenate(cols).astype(np.int32)
    data = np.concatenate(data).astype(np.float64)
    return sp.coo_matrix((data, (rows, cols)), shape=(n, n))

def normalize_columns_to_stochastic(A_coo):
    """Make columns sum to 1 (for PageRank-like power iteration)."""
    col_sums = np.array(A_coo.sum(axis=0)).ravel()
    col_sums_safe = np.where(col_sums == 0, 1.0, col_sums)
    data = A_coo.data * (1.0 / col_sums_safe[A_coo.col])
    return sp.coo_matrix((data, (A_coo.row, A_coo.col)), shape=A_coo.shape)

def coo_to_cdsf_blocks(coo, rows_block_size=4096):
    """
    Convert COO -> CDSF blocks:
      - group adjacent offsets into clusters
      - for each cluster, find min..max row and split into row blocks of size rows_block_size
      - produce list of clusters: each has 'offsets' array and list of blocks {'r0','r1','data'}
    """
    r = coo.row; c = coo.col; v = coo.data
    offsets = np.unique(c - r); offsets.sort()
    # group contiguous offsets into clusters
    clusters = []
    if offsets.size == 0:
        return []
    cur = [offsets[0]]
    for off in offsets[1:]:
        if off == cur[-1] + 1:
            cur.append(off)
        else:
            clusters.append(np.array(cur, dtype=np.int32))
            cur = [off]
    clusters.append(np.array(cur, dtype=np.int32))

    from collections import defaultdict
    off_map = defaultdict(list)
    for rr, cc, vv in zip(r, c, v):
        off_map[cc - rr].append((rr, vv))

    cdsf_clusters = []
    total_blocks = 0
    for off_arr in clusters:
        w = len(off_arr)
        rows_with = sorted({rr for off in off_arr for rr, _ in off_map.get(off, [])})
        if not rows_with:
            continue
        minr = rows_with[0]; maxr = rows_with[-1]
        blocks = []
        r0 = minr
        while r0 <= maxr:
            r1 = min(r0 + rows_block_size, maxr + 1)
            h = r1 - r0
            block_data = np.zeros((h, w), dtype=np.float64)
            # fill block_data (dense local block)
            for k, off in enumerate(off_arr):
                items = off_map.get(off, ())
                if not items:
                    continue
                for (rr, vv) in items:
                    if r0 <= rr < r1:
                        block_data[rr - r0, k] = vv
            if block_data.any():
                blocks.append({'r0': r0, 'r1': r1, 'data': block_data})
            r0 = r1
        total_blocks += len(blocks)
        cdsf_clusters.append({'offsets': off_arr, 'blocks': blocks})
    return cdsf_clusters, total_blocks

# Try to import Numba
numba_available = False
if use_numba:
    try:
        import numba as nb
        from numba.typed import List
        numba_available = True
    except Exception as e:
        print("Numba not available:", e)
        numba_available = False

# Numba kernel preparation helpers
if numba_available:
    from numba.typed import List

    def prepare_cdsf_for_numba(cdsf):
        """Convert cdsf clusters into numba-typed structures: typed list of offsets, arrays for blocks, typed list of 2D arrays for block data."""
        typed_clusters_offsets = List()
        clusters_block_counts = []
        blocks_r0 = []
        blocks_r1 = []
        typed_blocks_data = List()
        blocks_widths = []
        for cl in cdsf:
            offs = cl['offsets'].astype(np.int32)
            typed_clusters_offsets.append(offs)
            clusters_block_counts.append(len(cl['blocks']))
            for blk in cl['blocks']:
                blocks_r0.append(int(blk['r0']))
                blocks_r1.append(int(blk['r1']))
                bd = np.ascontiguousarray(blk['data'].astype(np.float64))
                typed_blocks_data.append(bd)
                blocks_widths.append(bd.shape[1])
        return typed_clusters_offsets, np.array(clusters_block_counts, dtype=np.int32), np.array(blocks_r0, dtype=np.int32), np.array(blocks_r1, dtype=np.int32), typed_blocks_data, np.array(blocks_widths, dtype=np.int32)

    @nb.njit(fastmath=True, nogil=True)
    def cdsf_matvec_numba(clusters_offsets_list, clusters_block_counts, blocks_r0, blocks_r1, blocks_data_list, blocks_widths, x, y_out):
        nloc = x.shape[0]
        # zero y
        for i in range(nloc):
            y_out[i] = 0.0
        block_idx = 0
        num_clusters = len(clusters_offsets_list)
        for ci in range(num_clusters):
            offs = clusters_offsets_list[ci]   # 1D int32
            num_blocks = clusters_block_counts[ci]
            w = offs.shape[0]
            for bi in range(num_blocks):
                r0 = blocks_r0[block_idx]; r1 = blocks_r1[block_idx]
                bd = blocks_data_list[block_idx]   # 2D array shape (r1-r0, w)
                h = r1 - r0
                for ii in range(h):
                    gi = r0 + ii
                    # unrolled inner: iterate kk
                    for kk in range(w):
                        a = bd[ii, kk]
                        if a != 0.0:
                            j = gi + offs[kk]
                            # bounds-check is safe but inexpensive compared to memory ops; keep it
                            if 0 <= j < nloc:
                                y_out[gi] += a * x[j]
                block_idx += 1
        return

    def make_cdsf_matvec_numba(cdsf):
        typed_clusters_offsets, clusters_block_counts, blocks_r0, blocks_r1, typed_blocks_data, blocks_widths = prepare_cdsf_for_numba(cdsf)
        def matvec(x):
            y = np.zeros_like(x)
            cdsf_matvec_numba(typed_clusters_offsets, clusters_block_counts, blocks_r0, blocks_r1, typed_blocks_data, blocks_widths, x, y)
            return y
        return matvec

else:
    # pure-Python fallback (vectorized where possible) - slower
    def make_cdsf_matvec_py(cdsf):
        def matvec(x):
            y = np.zeros_like(x)
            nloc = x.shape[0]
            for cl in cdsf:
                offs = cl['offsets']
                for blk in cl['blocks']:
                    r0 = blk['r0']; r1 = blk['r1']; bd = blk['data']
                    h, w = bd.shape
                    for kk, off in enumerate(offs):
                        col = bd[:, kk]
                        if off == 0:
                            y[r0:r1] += col * x[r0:r1]
                        elif off > 0:
                            limit = min(r1, nloc - off)
                            if r0 < limit:
                                y[r0:limit] += col[:(limit - r0)] * x[r0 + off: limit + off]
                        else:
                            o = -off
                            start = max(r0, o)
                            if start < r1:
                                y[start:r1] += col[(start - r0):(r1 - r0)] * x[start - o:r1 - o]
            return y
        return matvec

# ------------------ Build matrix and conversions ------------------
print("Building clustered COO matrix (n={}, clusters={}, width={}, density={})".format(n, num_clusters, cluster_width, density_per_diagonal))
t0 = time.perf_counter()
A_coo = build_clustered_coo(n, num_clusters, cluster_width, density_per_diagonal, rng)
A_coo = normalize_columns_to_stochastic(A_coo)
t_build = time.perf_counter() - t0
print("Built COO: nnz = {}, build time = {:.3f}s".format(A_coo.nnz, t_build))

# COO -> CSR conversion
t0 = time.perf_counter()
A_csr = A_coo.tocsr()
t_csr_conv = time.perf_counter() - t0
print("COO -> CSR conversion time: {:.6f}s".format(t_csr_conv))

# COO -> CDSF conversion (block-row clusters)
t0 = time.perf_counter()
cdsf, total_blocks = coo_to_cdsf_blocks(A_coo, rows_block_size=rows_block_size)
t_cdsf_conv = time.perf_counter() - t0
print("COO -> CDSF conversion time: {:.6f}s; clusters={}, total_blocks={}".format(t_cdsf_conv, len(cdsf), total_blocks))

# prepare CDSF matvec
if numba_available:
    print("Numba available: compiling CDSF matvec (first call will include compile time).")
    cdsf_matvec = make_cdsf_matvec_numba(cdsf)
    # warm-up compile call
    xw = np.ones(n, dtype=np.float64) / n
    _ = cdsf_matvec(xw)
else:
    print("Numba NOT available: falling back to Python CDSF matvec (will be slow).")
    cdsf_matvec = make_cdsf_matvec_py(cdsf)

# warm CSR matvec
_ = A_csr.dot(np.ones(n, dtype=np.float64))

# ------------------ PageRank iterations (timed) ------------------
def pagerank_csr(Acsr, alpha, iterations, x0=None):
    nloc = Acsr.shape[0]
    if x0 is None:
        x = np.ones(nloc, dtype=np.float64) / nloc
    else:
        x = x0.copy()
    teleport = (1.0 - alpha) / nloc
    tstart = time.perf_counter()
    for _ in range(iterations):
        x = alpha * (Acsr.dot(x)) + teleport
    return x, time.perf_counter() - tstart

def pagerank_cdsf(cdsf_matvec_func, alpha, iterations, x0=None):
    if x0 is None:
        x = np.ones(n, dtype=np.float64) / n
    else:
        x = x0.copy()
    teleport = (1.0 - alpha) / n
    tstart = time.perf_counter()
    for _ in range(iterations):
        y = cdsf_matvec_func(x)
        x = alpha * y + teleport
    return x, time.perf_counter() - tstart

print("Running CSR PageRank for {} iterations...".format(iterations))
x0 = np.ones(n, dtype=np.float64) / n
x_csr, t_csr_iters = pagerank_csr(A_csr, alpha, iterations, x0=x0)
print("CSR total iterations time: {:.6f}s (per-iter {:.9f}s)".format(t_csr_iters, t_csr_iters/iterations))

print("Running CDSF PageRank for {} iterations...".format(iterations))
x_cdsf, t_cdsf_iters = pagerank_cdsf(cdsf_matvec, alpha, iterations, x0=x0)
print("CDSF total iterations time: {:.6f}s (per-iter {:.9f}s)".format(t_cdsf_iters, t_cdsf_iters/iterations))

# verify correctness (should be near-zero)
diff = np.linalg.norm(x_csr - x_cdsf)
print("L2 difference between CSR and CDSF PageRank results: {:.6e}".format(diff))

# compute breakeven (if any)
per_csr = t_csr_iters / iterations
per_cdsf = t_cdsf_iters / iterations
if per_cdsf < per_csr:
    breakeven_iters = math.ceil((t_cdsf_conv - t_csr_conv) / (per_csr - per_cdsf))
else:
    breakeven_iters = None

# results summary and save CSV
summary = {
    "n": n,
    "nnz": int(A_coo.nnz),
    "clusters": len(cdsf),
    "total_blocks": total_blocks,
    "csr_conv_s": t_csr_conv,
    "cdsf_conv_s": t_cdsf_conv,
    "csr_iters_s": t_csr_iters,
    "cdsf_iters_s": t_cdsf_iters,
    "csr_per_iter_s": per_csr,
    "cdsf_per_iter_s": per_cdsf,
    "breakeven_iters": breakeven_iters,
    "numba_available": numba_available
}
df = pd.DataFrame(list(summary.items()), columns=["metric","value"])
csv_path = "./CDSF_vs_CSR_longrun_summary.csv"
df.to_csv(csv_path, index=False)
print("\nSummary (saved to {}):\n".format(csv_path), df.to_string(index=False))
if breakeven_iters is None:
    print("\nNo breakeven found (CDSF per-iteration slower than CSR for these parameters).")
else:
    print("\nBreakeven at ~{} iterations (i.e., beyond this many iterations CDSF total time < CSR total time).".format(breakeven_iters))
    print("Total time CSR: conv {:.4f} + iters*{:.6f}; CDSF: conv {:.4f} + iters*{:.6f}".format(t_csr_conv, per_csr, t_cdsf_conv, per_cdsf))

# End


Building clustered COO matrix (n=40000, clusters=12, width=24, density=0.6)
Built COO: nnz = 6892174, build time = 0.469s
COO -> CSR conversion time: 0.221116s
COO -> CDSF conversion time: 20.444444s; clusters=12, total_blocks=120
Numba available: compiling CDSF matvec (first call will include compile time).
Running CSR PageRank for 2000 iterations...
CSR total iterations time: 20.738701s (per-iter 0.010369351s)
Running CDSF PageRank for 2000 iterations...
CDSF total iterations time: 149.191162s (per-iter 0.074595581s)
L2 difference between CSR and CDSF PageRank results: 4.882085e-19

Summary (saved to ./CDSF_vs_CSR_longrun_summary.csv):
          metric       value
              n       40000
            nnz     6892174
       clusters          12
   total_blocks         120
     csr_conv_s    0.221116
    cdsf_conv_s   20.444444
    csr_iters_s   20.738701
   cdsf_iters_s  149.191162
 csr_per_iter_s    0.010369
cdsf_per_iter_s    0.074596
breakeven_iters        None
numba_available  

In [ ]:
 # Notebook cell: CDSF vs CSR breakeven plot (uses measured per-iteration times)
# Copy-paste whole cell into a Jupyter notebook and run.
import time, math, os
import numpy as np
import scipy.sparse as sp
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Parameters (tune to your hardware) ----------
n = 80000                  # matrix dimension (reduce if low RAM)
num_clusters = 8
cluster_width = 100
density_per_diagonal = 0.80
warmup_iters = 50           # iterations used to measure per-iteration time
measure_iters = 200         # iterations used to measure per-iteration time (averaged)
plot_iteration_range = [10, 50, 100, 200, 500, 1000, 2000, 5000]  # x-axis
rows_block_size = 1192      # block height for CDSF
rng_seed = 2025
use_numba = True
out_plot = "/mnt/data/cdsf_vs_csr_breakeven.png"
out_csv  = "/mnt/data/cdsf_vs_csr_breakeven.csv"
# -------------------------------------------------------

rng = np.random.default_rng(rng_seed)

# --- helper: build clustered COOs (dense local blocks) ---
def build_clustered_coo(n, num_clusters, cluster_width, density, rng):
    rows = []
    cols = []
    data = []
    spread = max(20, int(n / 200))
    centers = np.linspace(-spread, spread, num_clusters).astype(int)
    for center in centers:
        for k in range(cluster_width):
            offset = center - (cluster_width // 2) + k
            i_min = max(0, -offset)
            i_max = min(n, n - offset)
            length = i_max - i_min
            if length <= 0:
                continue
            mask = rng.random(length) < density
            if not mask.any():
                mask[rng.integers(0, length)] = True
            idxs = np.nonzero(mask)[0] + i_min
            jdxs = idxs + offset
            rows.append(idxs); cols.append(jdxs)
            data.append(rng.random(len(idxs)) + 0.1)
    rows = np.concatenate(rows).astype(np.int32)
    cols = np.concatenate(cols).astype(np.int32)
    data = np.concatenate(data).astype(np.float64)
    return sp.coo_matrix((data, (rows, cols)), shape=(n, n))

def normalize_columns_to_stochastic(A_coo):
    col_sums = np.array(A_coo.sum(axis=0)).ravel()
    col_sums_safe = np.where(col_sums == 0, 1.0, col_sums)
    data = A_coo.data * (1.0 / col_sums_safe[A_coo.col])
    return sp.coo_matrix((data, (A_coo.row, A_coo.col)), shape=A_coo.shape)

# --- CDSF conversion (block-row clusters) ---
def coo_to_cdsf_blocks(coo, rows_block_size=4096):
    r = coo.row; c = coo.col; v = coo.data
    offsets = np.unique(c - r); offsets.sort()
    if offsets.size == 0:
        return []
    clusters = []
    cur = [offsets[0]]
    for off in offsets[1:]:
        if off == cur[-1] + 1:
            cur.append(off)
        else:
            clusters.append(np.array(cur, dtype=np.int32)); cur = [off]
    clusters.append(np.array(cur, dtype=np.int32))

    from collections import defaultdict
    off_map = defaultdict(list)
    for rr, cc, vv in zip(r, c, v):
        off_map[cc - rr].append((rr, vv))

    cdsf_clusters = []
    total_blocks = 0
    for off_arr in clusters:
        w = len(off_arr)
        rows_with = sorted({rr for off in off_arr for rr, _ in off_map.get(off, [])})
        if not rows_with:
            continue
        minr = rows_with[0]; maxr = rows_with[-1]
        blocks = []
        r0 = minr
        while r0 <= maxr:
            r1 = min(r0 + rows_block_size, maxr + 1)
            h = r1 - r0
            block_data = np.zeros((h, w), dtype=np.float64)
            for k, off in enumerate(off_arr):
                for (rr, vv) in off_map.get(off, ()):
                    if r0 <= rr < r1:
                        block_data[rr - r0, k] = vv
            if block_data.any():
                blocks.append({'r0': r0, 'r1': r1, 'data': block_data})
            r0 = r1
        total_blocks += len(blocks)
        cdsf_clusters.append({'offsets': off_arr, 'blocks': blocks})
    return cdsf_clusters, total_blocks

# --- try numba ---
numba_available = False
if use_numba:
    try:
        import numba as nb
        from numba.typed import List
        numba_available = True
    except Exception:
        numba_available = False

# prepare numba kernel if available
if numba_available:
    from numba.typed import List
    def prepare_cdsf_for_numba(cdsf):
        typed_clusters_offsets = List()
        clusters_block_counts = []
        blocks_r0 = []; blocks_r1 = []
        typed_blocks_data = List(); blocks_widths = []
        for cl in cdsf:
            offs = cl['offsets'].astype(np.int32); typed_clusters_offsets.append(offs)
            clusters_block_counts.append(len(cl['blocks']))
            for blk in cl['blocks']:
                blocks_r0.append(int(blk['r0'])); blocks_r1.append(int(blk['r1']))
                bd = np.ascontiguousarray(blk['data'].astype(np.float64)); typed_blocks_data.append(bd)
                blocks_widths.append(bd.shape[1])
        return typed_clusters_offsets, np.array(clusters_block_counts, dtype=np.int32), np.array(blocks_r0, dtype=np.int32), np.array(blocks_r1, dtype=np.int32), typed_blocks_data, np.array(blocks_widths, dtype=np.int32)

    @nb.njit(fastmath=True, nogil=True)
    def cdsf_matvec_numba(clusters_offsets_list, clusters_block_counts, blocks_r0, blocks_r1, blocks_data_list, blocks_widths, x, y_out):
        nloc = x.shape[0]
        for ii in range(nloc):
            y_out[ii] = 0.0
        block_idx = 0
        num_clusters = len(clusters_offsets_list)
        for ci in range(num_clusters):
            offs = clusters_offsets_list[ci]
            num_blocks = clusters_block_counts[ci]
            w = offs.shape[0]
            for bi in range(num_blocks):
                r0 = blocks_r0[block_idx]; r1 = blocks_r1[block_idx]
                bd = blocks_data_list[block_idx]
                h = r1 - r0
                for ii in range(h):
                    gi = r0 + ii
                    for kk in range(w):
                        a = bd[ii, kk]
                        if a != 0.0:
                            j = gi + offs[kk]
                            if 0 <= j < nloc:
                                y_out[gi] += a * x[j]
                block_idx += 1
        return

    def make_cdsf_matvec_numba(cdsf):
        typed_clusters_offsets, clusters_block_counts, blocks_r0, blocks_r1, typed_blocks_data, blocks_widths = prepare_cdsf_for_numba(cdsf)
        def matvec(x):
            y = np.zeros_like(x)
            cdsf_matvec_numba(typed_clusters_offsets, clusters_block_counts, blocks_r0, blocks_r1, typed_blocks_data, blocks_widths, x, y)
            return y
        return matvec
else:
    # python fallback matvec
    def make_cdsf_matvec_py(cdsf):
        def matvec(x):
            y = np.zeros_like(x)
            nloc = x.shape[0]
            for cl in cdsf:
                offs = cl['offsets']
                for blk in cl['blocks']:
                    r0 = blk['r0']; r1 = blk['r1']; bd = blk['data']
                    h, w = bd.shape
                    for kk, off in enumerate(offs):
                        col = bd[:, kk]
                        if off == 0:
                            y[r0:r1] += col * x[r0:r1]
                        elif off > 0:
                            limit = min(r1, nloc - off)
                            if r0 < limit:
                                y[r0:limit] += col[:(limit - r0)] * x[r0 + off: limit + off]
                        else:
                            o = -off
                            start = max(r0, o)
                            if start < r1:
                                y[start:r1] += col[(start - r0):(r1 - r0)] * x[start - o:r1 - o]
            return y
        return matvec

# ---------- Build and convert ----------
print("Building matrix n={}, clusters={}, width={}, density={}".format(n, num_clusters, cluster_width, density_per_diagonal))
t0 = time.perf_counter()
A_coo = build_clustered_coo(n, num_clusters, cluster_width, density_per_diagonal, rng)
A_coo = normalize_columns_to_stochastic(A_coo)
t_build = time.perf_counter() - t0
print("Built COO: nnz = {}, build time = {:.3f}s".format(A_coo.nnz, t_build))

# COO->CSR
t0 = time.perf_counter()
A_csr = A_coo.tocsr()
t_csr_conv = time.perf_counter() - t0
print("COO->CSR conv: {:.4f}s".format(t_csr_conv))

# COO->CDSF
t0 = time.perf_counter()
cdsf, total_blocks = coo_to_cdsf_blocks(A_coo, rows_block_size=rows_block_size)
t_cdsf_conv = time.perf_counter() - t0
print("COO->CDSF conv: {:.4f}s clusters={} blocks={}".format(t_cdsf_conv, len(cdsf), total_blocks))

# prepare matvecs
if numba_available:
    print("Numba available: preparing CDSF matvec (compilation may occur on first call).")
    cdsf_matvec = make_cdsf_matvec_numba(cdsf)
    # warm compile
    _ = cdsf_matvec(np.ones(n, dtype=np.float64) / n)
else:
    print("Numba NOT available: using Python CDSF matvec.")
    cdsf_matvec = make_cdsf_matvec_py(cdsf)

# warm CSR
_ = A_csr.dot(np.ones(n, dtype=np.float64))

# ---------- Measure per-iteration time (CSR and CDSF) ----------
def measure_per_iter_csr(Acsr, iters):
    x = np.ones(Acsr.shape[0], dtype=np.float64) / Acsr.shape[0]
    # warm
    _ = Acsr.dot(x)
    t0 = time.perf_counter()
    for _ in range(iters):
        x = Acsr.dot(x)
    t = time.perf_counter() - t0
    return t / iters

def measure_per_iter_cdsf(cdsf_matvec_func, iters):
    x = np.ones(n, dtype=np.float64) / n
    # warm
    _ = cdsf_matvec_func(x)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = cdsf_matvec_func(x)
    t = time.perf_counter() - t0
    return t / iters

print("Measuring per-iteration times ({} iters measurement)...".format(measure_iters))
per_csr = measure_per_iter_csr(A_csr, measure_iters)
per_cdsf = measure_per_iter_cdsf(cdsf_matvec, measure_iters)
print("Measured per-iter: CSR {:.6e}s, CDSF {:.6e}s".format(per_csr, per_cdsf))

# ---------- Compute total time curves for a range of iteration counts ----------
iters_range = np.array(plot_iteration_range, dtype=int)
total_time_csr = t_csr_conv + per_csr * iters_range
total_time_cdsf = t_cdsf_conv + per_cdsf * iters_range

# compute breakeven analytically if possible
breakeven_iters = None
if per_cdsf < per_csr:
    breakeven_iters = math.ceil((t_cdsf_conv - t_csr_conv) / (per_csr - per_cdsf))
    print("Analytic breakeven iterations:", breakeven_iters)
else:
    print("CDSF per-iteration is NOT faster than CSR (no breakeven).")

# ---------- Plot ----------
plt.figure(figsize=(8,5))
plt.plot(iters_range, total_time_csr, '-o', label='CSR (conv + iters * per_iter)')
plt.plot(iters_range, total_time_cdsf, '-o', label='CDSF (conv + iters * per_iter)')
if breakeven_iters is not None:
    # compute y at breakeven and plot vertical line
    yb = t_csr_conv + per_csr * breakeven_iters
    plt.axvline(breakeven_iters, color='gray', linestyle='--', label=f'Breakeven ≈ {breakeven_iters}')
    plt.scatter([breakeven_iters],[yb], color='black')
plt.xlabel('Number of iterations')
plt.ylabel('Total time (seconds)')
plt.title('CSR vs CDSF: total time vs iterations (estimated from measured per-iter)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(out_plot, dpi=150)
print("Plot saved to:", out_plot)

# ---------- Save numeric summary ----------
summary = {
    'n': n, 'nnz': int(A_coo.nnz), 'clusters': len(cdsf), 'blocks': total_blocks,
    'csr_conv_s': t_csr_conv, 'cdsf_conv_s': t_cdsf_conv,
    'csr_per_iter_s': per_csr, 'cdsf_per_iter_s': per_cdsf,
    'breakeven_iters': breakeven_iters
}
df = pd.DataFrame(list(summary.items()), columns=['metric','value'])
df.to_csv(out_csv, index=False)
print("CSV saved to:", out_csv)
print("\nSummary:\n", df)

# also print arrays for quick inspection
print("\nIterations:", iters_range)
print("CSR total times:", total_time_csr)
print("CDSF total times:", total_time_cdsf)


In [ ]:
 # Notebook cell: CDSF vs CSR breakeven plot (uses measured per-iteration times)
# Copy-paste whole cell into a Jupyter notebook and run.
import time, math, os
import numpy as np
import scipy.sparse as sp
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Parameters (tune to your hardware) ----------
n = 80000                  # matrix dimension (reduce if low RAM)
num_clusters = 8
cluster_width = 100
density_per_diagonal = 0.80
warmup_iters = 50           # iterations used to measure per-iteration time
measure_iters = 200         # iterations used to measure per-iteration time (averaged)
plot_iteration_range = [10, 50, 100, 200, 500, 1000, 2000, 5000]  # x-axis
rows_block_size = 1192      # block height for CDSF
rng_seed = 2025
use_numba = True
out_plot = "/mnt/data/cdsf_vs_csr_breakeven.png"
out_csv  = "/mnt/data/cdsf_vs_csr_breakeven.csv"
# -------------------------------------------------------

rng = np.random.default_rng(rng_seed)

# --- helper: build clustered COOs (dense local blocks) ---
def build_clustered_coo(n, num_clusters, cluster_width, density, rng):
    rows = []
    cols = []
    data = []
    spread = max(20, int(n / 200))
    centers = np.linspace(-spread, spread, num_clusters).astype(int)
    for center in centers:
        for k in range(cluster_width):
            offset = center - (cluster_width // 2) + k
            i_min = max(0, -offset)
            i_max = min(n, n - offset)
            length = i_max - i_min
            if length <= 0:
                continue
            mask = rng.random(length) < density
            if not mask.any():
                mask[rng.integers(0, length)] = True
            idxs = np.nonzero(mask)[0] + i_min
            jdxs = idxs + offset
            rows.append(idxs); cols.append(jdxs)
            data.append(rng.random(len(idxs)) + 0.1)
    rows = np.concatenate(rows).astype(np.int32)
    cols = np.concatenate(cols).astype(np.int32)
    data = np.concatenate(data).astype(np.float64)
    return sp.coo_matrix((data, (rows, cols)), shape=(n, n))

def normalize_columns_to_stochastic(A_coo):
    col_sums = np.array(A_coo.sum(axis=0)).ravel()
    col_sums_safe = np.where(col_sums == 0, 1.0, col_sums)
    data = A_coo.data * (1.0 / col_sums_safe[A_coo.col])
    return sp.coo_matrix((data, (A_coo.row, A_coo.col)), shape=A_coo.shape)

# --- CDSF conversion (block-row clusters) ---
def coo_to_cdsf_blocks(coo, rows_block_size=4096):
    r = coo.row; c = coo.col; v = coo.data
    offsets = np.unique(c - r); offsets.sort()
    if offsets.size == 0:
        return []
    clusters = []
    cur = [offsets[0]]
    for off in offsets[1:]:
        if off == cur[-1] + 1:
            cur.append(off)
        else:
            clusters.append(np.array(cur, dtype=np.int32)); cur = [off]
    clusters.append(np.array(cur, dtype=np.int32))

    from collections import defaultdict
    off_map = defaultdict(list)
    for rr, cc, vv in zip(r, c, v):
        off_map[cc - rr].append((rr, vv))

    cdsf_clusters = []
    total_blocks = 0
    for off_arr in clusters:
        w = len(off_arr)
        rows_with = sorted({rr for off in off_arr for rr, _ in off_map.get(off, [])})
        if not rows_with:
            continue
        minr = rows_with[0]; maxr = rows_with[-1]
        blocks = []
        r0 = minr
        while r0 <= maxr:
            r1 = min(r0 + rows_block_size, maxr + 1)
            h = r1 - r0
            block_data = np.zeros((h, w), dtype=np.float64)
            for k, off in enumerate(off_arr):
                for (rr, vv) in off_map.get(off, ()):
                    if r0 <= rr < r1:
                        block_data[rr - r0, k] = vv
            if block_data.any():
                blocks.append({'r0': r0, 'r1': r1, 'data': block_data})
            r0 = r1
        total_blocks += len(blocks)
        cdsf_clusters.append({'offsets': off_arr, 'blocks': blocks})
    return cdsf_clusters, total_blocks

# --- try numba ---
numba_available = False
if use_numba:
    try:
        import numba as nb
        from numba.typed import List
        numba_available = True
    except Exception:
        numba_available = False

# prepare numba kernel if available
if numba_available:
    from numba.typed import List
    def prepare_cdsf_for_numba(cdsf):
        typed_clusters_offsets = List()
        clusters_block_counts = []
        blocks_r0 = []; blocks_r1 = []
        typed_blocks_data = List(); blocks_widths = []
        for cl in cdsf:
            offs = cl['offsets'].astype(np.int32); typed_clusters_offsets.append(offs)
            clusters_block_counts.append(len(cl['blocks']))
            for blk in cl['blocks']:
                blocks_r0.append(int(blk['r0'])); blocks_r1.append(int(blk['r1']))
                bd = np.ascontiguousarray(blk['data'].astype(np.float64)); typed_blocks_data.append(bd)
                blocks_widths.append(bd.shape[1])
        return typed_clusters_offsets, np.array(clusters_block_counts, dtype=np.int32), np.array(blocks_r0, dtype=np.int32), np.array(blocks_r1, dtype=np.int32), typed_blocks_data, np.array(blocks_widths, dtype=np.int32)

    @nb.njit(fastmath=True, nogil=True)
    def cdsf_matvec_numba(clusters_offsets_list, clusters_block_counts, blocks_r0, blocks_r1, blocks_data_list, blocks_widths, x, y_out):
        nloc = x.shape[0]
        for ii in range(nloc):
            y_out[ii] = 0.0
        block_idx = 0
        num_clusters = len(clusters_offsets_list)
        for ci in range(num_clusters):
            offs = clusters_offsets_list[ci]
            num_blocks = clusters_block_counts[ci]
            w = offs.shape[0]
            for bi in range(num_blocks):
                r0 = blocks_r0[block_idx]; r1 = blocks_r1[block_idx]
                bd = blocks_data_list[block_idx]
                h = r1 - r0
                for ii in range(h):
                    gi = r0 + ii
                    for kk in range(w):
                        a = bd[ii, kk]
                        if a != 0.0:
                            j = gi + offs[kk]
                            if 0 <= j < nloc:
                                y_out[gi] += a * x[j]
                block_idx += 1
        return

    def make_cdsf_matvec_numba(cdsf):
        typed_clusters_offsets, clusters_block_counts, blocks_r0, blocks_r1, typed_blocks_data, blocks_widths = prepare_cdsf_for_numba(cdsf)
        def matvec(x):
            y = np.zeros_like(x)
            cdsf_matvec_numba(typed_clusters_offsets, clusters_block_counts, blocks_r0, blocks_r1, typed_blocks_data, blocks_widths, x, y)
            return y
        return matvec
else:
    # python fallback matvec
    def make_cdsf_matvec_py(cdsf):
        def matvec(x):
            y = np.zeros_like(x)
            nloc = x.shape[0]
            for cl in cdsf:
                offs = cl['offsets']
                for blk in cl['blocks']:
                    r0 = blk['r0']; r1 = blk['r1']; bd = blk['data']
                    h, w = bd.shape
                    for kk, off in enumerate(offs):
                        col = bd[:, kk]
                        if off == 0:
                            y[r0:r1] += col * x[r0:r1]
                        elif off > 0:
                            limit = min(r1, nloc - off)
                            if r0 < limit:
                                y[r0:limit] += col[:(limit - r0)] * x[r0 + off: limit + off]
                        else:
                            o = -off
                            start = max(r0, o)
                            if start < r1:
                                y[start:r1] += col[(start - r0):(r1 - r0)] * x[start - o:r1 - o]
            return y
        return matvec

# ---------- Build and convert ----------
print("Building matrix n={}, clusters={}, width={}, density={}".format(n, num_clusters, cluster_width, density_per_diagonal))
t0 = time.perf_counter()
A_coo = build_clustered_coo(n, num_clusters, cluster_width, density_per_diagonal, rng)
A_coo = normalize_columns_to_stochastic(A_coo)
t_build = time.perf_counter() - t0
print("Built COO: nnz = {}, build time = {:.3f}s".format(A_coo.nnz, t_build))

# COO->CSR
t0 = time.perf_counter()
A_csr = A_coo.tocsr()
t_csr_conv = time.perf_counter() - t0
print("COO->CSR conv: {:.4f}s".format(t_csr_conv))

# COO->CDSF
t0 = time.perf_counter()
cdsf, total_blocks = coo_to_cdsf_blocks(A_coo, rows_block_size=rows_block_size)
t_cdsf_conv = time.perf_counter() - t0
print("COO->CDSF conv: {:.4f}s clusters={} blocks={}".format(t_cdsf_conv, len(cdsf), total_blocks))

# prepare matvecs
if numba_available:
    print("Numba available: preparing CDSF matvec (compilation may occur on first call).")
    cdsf_matvec = make_cdsf_matvec_numba(cdsf)
    # warm compile
    _ = cdsf_matvec(np.ones(n, dtype=np.float64) / n)
else:
    print("Numba NOT available: using Python CDSF matvec.")
    cdsf_matvec = make_cdsf_matvec_py(cdsf)

# warm CSR
_ = A_csr.dot(np.ones(n, dtype=np.float64))

# ---------- Measure per-iteration time (CSR and CDSF) ----------
def measure_per_iter_csr(Acsr, iters):
    x = np.ones(Acsr.shape[0], dtype=np.float64) / Acsr.shape[0]
    # warm
    _ = Acsr.dot(x)
    t0 = time.perf_counter()
    for _ in range(iters):
        x = Acsr.dot(x)
    t = time.perf_counter() - t0
    return t / iters

def measure_per_iter_cdsf(cdsf_matvec_func, iters):
    x = np.ones(n, dtype=np.float64) / n
    # warm
    _ = cdsf_matvec_func(x)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = cdsf_matvec_func(x)
    t = time.perf_counter() - t0
    return t / iters

print("Measuring per-iteration times ({} iters measurement)...".format(measure_iters))
per_csr = measure_per_iter_csr(A_csr, measure_iters)
per_cdsf = measure_per_iter_cdsf(cdsf_matvec, measure_iters)
print("Measured per-iter: CSR {:.6e}s, CDSF {:.6e}s".format(per_csr, per_cdsf))

# ---------- Compute total time curves for a range of iteration counts ----------
iters_range = np.array(plot_iteration_range, dtype=int)
total_time_csr = t_csr_conv + per_csr * iters_range
total_time_cdsf = t_cdsf_conv + per_cdsf * iters_range

# compute breakeven analytically if possible
breakeven_iters = None
if per_cdsf < per_csr:
    breakeven_iters = math.ceil((t_cdsf_conv - t_csr_conv) / (per_csr - per_cdsf))
    print("Analytic breakeven iterations:", breakeven_iters)
else:
    print("CDSF per-iteration is NOT faster than CSR (no breakeven).")

# ---------- Plot ----------
plt.figure(figsize=(8,5))
plt.plot(iters_range, total_time_csr, '-o', label='CSR (conv + iters * per_iter)')
plt.plot(iters_range, total_time_cdsf, '-o', label='CDSF (conv + iters * per_iter)')
if breakeven_iters is not None:
    # compute y at breakeven and plot vertical line
    yb = t_csr_conv + per_csr * breakeven_iters
    plt.axvline(breakeven_iters, color='gray', linestyle='--', label=f'Breakeven ≈ {breakeven_iters}')
    plt.scatter([breakeven_iters],[yb], color='black')
plt.xlabel('Number of iterations')
plt.ylabel('Total time (seconds)')
plt.title('CSR vs CDSF: total time vs iterations (estimated from measured per-iter)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(out_plot, dpi=150)
print("Plot saved to:", out_plot)

# ---------- Save numeric summary ----------
summary = {
    'n': n, 'nnz': int(A_coo.nnz), 'clusters': len(cdsf), 'blocks': total_blocks,
    'csr_conv_s': t_csr_conv, 'cdsf_conv_s': t_cdsf_conv,
    'csr_per_iter_s': per_csr, 'cdsf_per_iter_s': per_cdsf,
    'breakeven_iters': breakeven_iters
}
df = pd.DataFrame(list(summary.items()), columns=['metric','value'])
df.to_csv(out_csv, index=False)
print("CSV saved to:", out_csv)
print("\nSummary:\n", df)

# also print arrays for quick inspection
print("\nIterations:", iters_range)
print("CSR total times:", total_time_csr)
print("CDSF total times:", total_time_cdsf)
